# Episia : Epidemiology & Biostatistics for Python

**Xcept-Health · Burkina Faso**  
*Open-source epidemiology toolkit built for African public health contexts.*

---

This notebook demonstrates the core capabilities of Episia:

1. Compartmental epidemic models (SIR / SEIR / SEIRD)
2. Biostatistics  risk ratios, odds ratios, diagnostic tests
3. Surveillance data  alert detection, endemic channel
4. Sample size calculation
5. ROC curve analysis
6. Scenario comparison
7. Automated report generation

In [25]:
# Install if needed
# !pip install episia

In [26]:
import numpy as np
import pandas as pd
from episia import epi, __version__

print(f"Episia v{__version__}  ready")

Episia v0.1.0  ready


---
## 1. SEIR epidemic model

Modelling a respiratory outbreak in a population of 1 million.  
Parameters calibrated on a COVID-19-like scenario.

In [47]:
from episia.models import SEIRModel
from episia.models.parameters import SEIRParameters

params = SEIRParameters(
    N     = 1_000_000,
    I0    = 10,
    E0    = 50,
    beta  = 0.35,       # transmission rate
    sigma = 1 / 5.2,    # 1 / incubation period (days)
    gamma = 1 / 14,     # 1 / infectious period (days)
    t_span = (0, 365),
)

result = SEIRModel(params).run()
print(result)

SEIR Model
  R₀          : 4.900
  Peak infected : 331,751  at t=84.5
  Final size    : 99.2%
  Duration      : 0–365


Risk Ratio: 2.667 (1.514-4.696) *


True

In [28]:
# Interactive Plotly trajectory
result.plot().show()

---
## 2. SEIRD model  with mortality

In [29]:
from episia.models import SEIRDModel
from episia.models.parameters import SEIRDParameters

params_d = SEIRDParameters(
    N     = 1_000_000,
    I0    = 10,
    E0    = 50,
    beta  = 0.35,
    sigma = 1 / 5.2,
    gamma = 0.09,       # recovery rate
    mu    = 0.01,       # disease-induced mortality rate  ->  CFR ~10%
    t_span = (0, 365),
)

result_d = SEIRDModel(params_d).run()
print(result_d)
print(f"\nTotal deaths: {result_d.compartments['D'][-1]:,.0f}")

SEIRD Model
  R₀          : 3.500
  Peak infected : 228,359  at t=92.5
  Final size    : 86.9%
  Duration      : 0–365

Total deaths: 96,599


In [30]:
result_d.plot().show()

---
## 3. Scenario comparison  intervention impact

In [31]:
from episia.models import ScenarioRunner
from episia.models.parameters import ScenarioSet

scenarios = ScenarioSet([
    ("No intervention", SEIRParameters(N=1_000_000, I0=10, E0=50, beta=0.35, sigma=1/5.2, gamma=1/14)),
    ("50% reduction",   SEIRParameters(N=1_000_000, I0=10, E0=50, beta=0.18, sigma=1/5.2, gamma=1/14)),
    ("70% reduction",   SEIRParameters(N=1_000_000, I0=10, E0=50, beta=0.11, sigma=1/5.2, gamma=1/14)),
])

sr = ScenarioRunner(SEIRModel).run(scenarios)
sr.plot(compartment="I").show()

In [32]:
# Tabular summary
sr.to_dataframe()

,r0,peak_infected,peak_time,final_size
scenario,,,,
No intervention,4.90,331752.966066,84.552846,0.987040
50% reduction,2.52,170535.828903,160.000000,0.410183
70% reduction,1.54,2213.692382,160.000000,0.006128


---
## 4. Biostatistics  association measures

In [33]:
from episia.stats.contingency import Table2x2, risk_ratio, odds_ratio

# 2x2 table:
#              Cases   Non-cases
# Exposed        40       20
# Unexposed      10       30

table = Table2x2(a=40, b=10, c=20, d=30)

rr  = table.risk_ratio()
or_ = table.odds_ratio()
rd  = table.risk_difference()

print(rr)
print(or_)
print(f"Risk difference: {rd['estimate']:.3f} ({rd['ci_lower']:.3f}–{rd['ci_upper']:.3f})")
print(f"Significant: {rr.significant}")
print(f"p-value: {rr.p_value:.4f}")

Risk Ratio: 2.667 (1.514-4.696) *
Odds Ratio: 6.000 (2.453-14.678) *
Risk difference: 0.417 (0.237–0.596)
Significant: True
p-value: 0.0007


In [34]:
# Full summary
summary = table.summary()
pd.DataFrame({
    "Measure": ["Risk exposed", "Risk unexposed", "RR", "OR", "PAF"],
    "Value":   [
        f"{summary['risks']['exposed']:.3f}",
        f"{summary['risks']['unexposed']:.3f}",
        f"{summary['risk_ratio']['estimate']:.3f}",
        f"{summary['odds_ratio']['estimate']:.3f}",
        f"{summary['attributable_fractions']['population']:.3f}",
    ]
})

,Measure,Value
0,Risk exposed,0.667
1,Risk unexposed,0.250
2,RR,2.667
3,OR,6.000
4,PAF,0.571


---
## 5. Proportion confidence interval

In [35]:
from episia.stats.descriptive import proportion_ci, CI_Method

methods = [CI_Method.WILSON, CI_Method.CLOPPER_PEARSON,
           CI_Method.JEFFREYS, CI_Method.AGRESTI_COULL]

rows = []
for m in methods:
    r = proportion_ci(k=45, n=200, method=m)
    rows.append({
        "Method":   r.method,
        "Proportion": f"{r.proportion:.4f}",
        "95% CI":   f"{r.ci_lower:.4f} – {r.ci_upper:.4f}",
        "Width":    f"{r.ci_upper - r.ci_lower:.4f}",
    })

pd.DataFrame(rows)

,Method,Proportion,95% CI,Width
0,wilson,0.2250,0.1726 – 0.2877,0.1151
1,clopper_pearson,0.2250,0.1691 – 0.2892,0.1201
2,jeffreys,0.2250,0.1713 – 0.2866,0.1152
3,agresti_coull,0.2250,0.1724 – 0.2880,0.1156


---
## 6. Diagnostic test evaluation

In [36]:
from episia.stats.diagnostic import diagnostic_test_2x2

# Malaria RDT evaluation
diag = diagnostic_test_2x2(tp=85, fp=8, fn=15, tn=92)
print(diag)
print(f"\nLR+: {diag.lr_positive:.2f}  (> 10 = excellent)")
print(f"LR-: {diag.lr_negative:.3f}  (< 0.1 = excellent)")

Sens: 0.850, Spec: 0.920, Acc: 0.885

LR+: 10.63  (> 10 = excellent)
LR-: 0.163  (< 0.1 = excellent)


---
## 7. ROC curve analysis

In [37]:
from episia.stats.diagnostic import roc_analysis

rng = np.random.default_rng(42)
n   = 200

# Simulate a diagnostic score  cases have higher scores
y_true  = np.array([1]*100 + [0]*100)
y_score = np.concatenate([
    rng.beta(5, 2, 100),   # cases  skewed high
    rng.beta(2, 5, 100),   # controls  skewed low
])

roc = roc_analysis(y_true, y_score)
print(roc)
print(f"\nOptimal threshold: {roc.optimal_threshold:.3f}")
print(f"At threshold  Sensitivity: {roc.optimal_point['sensitivity']:.3f}, "
      f"Specificity: {roc.optimal_point['specificity']:.3f}")

ROC Curve  AUC: 0.967
Optimal threshold: 0.564  (Sens=0.880, Spec=0.960)

Optimal threshold: 0.564
At threshold  Sensitivity: 0.880, Specificity: 0.960


In [38]:
roc.plot().show()

---
## 8. Sample size calculation

In [40]:
from episia.stats.samplesize import sample_size_risk_ratio, sample_size_single_proportion

# Cohort study  detect RR = 2.0
ss_cohort = sample_size_risk_ratio(
    rr_expected = 2.0,
    p0          = 0.10,
    alpha       = 0.05,
    power       = 0.80,
)
print('Cohort study:')
print(ss_cohort)

print()

# Prevalence survey
ss_survey = sample_size_single_proportion(
    expected_proportion = 0.15,   # expected prevalence
    precision           = 0.05,   # desired half-width of CI
    alpha               = 0.05,
)
print('Prevalence survey:')
print(ss_survey)


Cohort study:
Sample size: 199 per group (total: 398)

Prevalence survey:


---
## 9. Surveillance data  meningitis, Burkina Faso 2024

In [49]:
from episia.data.surveillance import SurveillanceDataset, AlertEngine
from episia.viz.curves import plot_epicurve

ds = SurveillanceDataset.from_csv(
    "data/meningitis_2024.csv",
    date_col       = "date",
    cases_col      = "cases",
    deaths_col     = "deaths",
    district_col   = "district",
    population_col = "population",
)

print(ds)
print()

summary = ds.summary()
pd.Series(summary)

SurveillanceDataset(n=52, cases=539, 2024-01-07 00:00:00 → 2024-12-29 00:00:00)



n_records                       52
total_cases                    539
date_start     2024-01-07 00:00:00
date_end       2024-12-29 00:00:00
n_districts                      1
districts                 [Centre]
dtype: object

In [50]:
# Alert detection
engine  = AlertEngine(ds)
alerts  = engine.run(threshold=15, zscore_threshold=2.0, use_endemic_channel=True)
summary = engine.alert_summary(alerts)

print(f"{summary['n_alerts']} alerts: {summary['severity_counts']}")

# Show top 5
for a in sorted(alerts, key=lambda x: x.severity, reverse=True)[:5]:
    print(f"  [{a.severity.upper():8s}]  {str(a.period)[:10]}  {a.message}")

16 alerts: {'alert': 8, 'epidemic': 5, 'warning': 3}
  [WARNING ]  2024-03-04  Z-score=2.48 ≥ 2.0
  [WARNING ]  2024-03-18  Z-score=2.78 ≥ 2.0
  [WARNING ]  2024-03-25  Z-score=2.08 ≥ 2.0
  [EPIDEMIC]  2024-03-04  35 cas ≥ seuil 15
  [EPIDEMIC]  2024-03-11  42 cas ≥ seuil 15


In [43]:
# Epidemic curve
ts = ds.to_timeseries_result()
plot_epicurve(ts, title="Meningitis  Burkina Faso 2024").show()

---
## 10. Automated report generation

In [51]:
from episia import EpiReport

report = EpiReport(
    title       = "Weekly Epidemiological Bulletin  Week 12",
    author      = "Dr. Ariel Shadrac Ouedraogo",
    institution = "Xcept-Health",
)

report.add_text(
    "This bulletin covers meningitis surveillance for the week of March 17–23, 2025.",
    title="Summary",
)

report.add_metrics({
    "Total cases":     summary.get('total_cases', 978),
    "Total deaths":    summary.get('total_deaths', 15),
    "CFR":             "1.5%",
    "Alert districts": summary['n_alerts'],
    "Districts": 5,
})

fig = plot_epicurve(ts, title="Meningitis  epidemic curve")
report.add_figure(fig, title="Weekly cases",
                  caption="Meningitis cases by week, Burkina Faso 2024.")
report.add_divider()

path = report.save_html("bulletin_w12.html")
print(f"Report saved → {path}")

Report saved → bulletin_w12.html


In [52]:
# Also export as model report
from episia import report_from_model

model_report = report_from_model(
    result,
    title       = "SEIR Analysis  Burkina Faso 2024",
    author      = "Dr. Ariel Shadrac Ouedraogo",
    institution = "Xcept-Health",
)
model_report.save_html("seir_report.html")
print("Model report saved → seir_report.html")

Model report saved → seir_report.html


---
## 11. Mantel-Haenszel  stratified analysis

In [46]:
from episia.stats.stratified import mantel_haenszel_or

# 3 strata (age groups)
strata = [
    Table2x2(15, 5,  25, 20),   # < 5 years
    Table2x2(30, 10, 40, 35),   # 5–14 years
    Table2x2(20, 8,  30, 25),   # 15+ years
]

mh = mantel_haenszel_or(strata)
print(mh)

Mantel-Haenszel OR: 2.384 (1.576-3.607)


---

## Summary

| Module | What was demonstrated |
|---|---|
| `episia.models` | SIR, SEIR, SEIRD, scenario comparison |
| `episia.stats.contingency` | Table2x2, RR, OR, RD, chi-square, PAF |
| `episia.stats.descriptive` | Proportion CI  4 methods |
| `episia.stats.diagnostic` | Sensitivity, specificity, LR+/-, ROC |
| `episia.stats.samplesize` | Cohort study, prevalence survey |
| `episia.stats.stratified` | Mantel-Haenszel pooled OR |
| `episia.data` | SurveillanceDataset, AlertEngine |
| `episia.viz` | plot_epicurve, ROC plot, model trajectories |
| `episia.api` | EpiReport, report_from_model |

---

*Episia  Built with precision for African public health · Xcept-Health · Burkina Faso*